# Semantic Scoring in Colab

This notebook reads `semantic_input.csv` (exported from local `plot-shift`), runs transformer sentiment scoring on GPU, and writes `semantic_scores.csv` ready for `plot-shift import-semantic-scores`.

Expected input columns:
- `imdb_id_int`
- `sentence_idx`
- `sentence_text`

Output columns:
- `imdb_id_int`
- `sentence_idx`
- `score`
- `model`
- `dimension`

In [ ]:
# Install runtime dependencies in Colab
!pip install -q transformers torch pandas

In [ ]:
from pathlib import Path

import pandas as pd

# Change these paths to your Colab/Drive paths
INPUT_CSV = "/content/semantic_input.csv"
OUTPUT_CSV = "/content/semantic_scores.csv"

# Free and strong default
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# Tune for your Colab runtime
BATCH_SIZE = 64
MAX_LENGTH = 256
DEVICE = 0  # 0 for GPU, -1 for CPU

input_path = Path(INPUT_CSV)
if not input_path.exists():
    raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")

df = pd.read_csv(INPUT_CSV)
required_cols = {"imdb_id_int", "sentence_idx", "sentence_text"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required input columns: {sorted(missing)}")

print(f"Loaded rows: {len(df)}")
df.head()

In [ ]:
from transformers import pipeline


def _label_to_key(label: str) -> str:
    value = label.strip().lower()
    if "neg" in value:
        return "negative"
    if "pos" in value:
        return "positive"
    if "neu" in value:
        return "neutral"
    return value


def _continuous_score(prediction):
    # top_k=None typically returns full per-label distribution.
    if isinstance(prediction, list):
        by_label = {}
        for item in prediction:
            by_label[_label_to_key(str(item.get("label", "")))] = float(item.get("score", 0.0))

        if "positive" in by_label and "negative" in by_label:
            return max(-1.0, min(1.0, by_label["positive"] - by_label["negative"]))

        if len(prediction) >= 2:
            first = float(prediction[0].get("score", 0.0))
            last = float(prediction[-1].get("score", 0.0))
            return max(-1.0, min(1.0, last - first))

        return 0.0

    # Fallback if model returns top label only.
    if isinstance(prediction, dict):
        label = _label_to_key(str(prediction.get("label", "")))
        score = float(prediction.get("score", 0.0))
        if label == "positive":
            return score
        if label == "negative":
            return -score

    return 0.0


classifier = pipeline(
    "text-classification",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=DEVICE,
    top_k=None,
)

print("Model loaded:", MODEL_NAME)

In [ ]:
rows_out = []
texts = df["sentence_text"].astype(str).tolist()

for start in range(0, len(texts), BATCH_SIZE):
    chunk_texts = texts[start : start + BATCH_SIZE]
    preds = classifier(chunk_texts, truncation=True, max_length=MAX_LENGTH)

    chunk_df = df.iloc[start : start + BATCH_SIZE]
    for (_, row), pred in zip(chunk_df.iterrows(), preds, strict=True):
        rows_out.append(
            {
                "imdb_id_int": int(row["imdb_id_int"]),
                "sentence_idx": int(row["sentence_idx"]),
                "score": float(_continuous_score(pred)),
                "model": f"semantic:{MODEL_NAME}",
                "dimension": "valence",
            }
        )

out_df = pd.DataFrame(rows_out)
out_df.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(out_df)} rows to {OUTPUT_CSV}")
out_df.head()

In [ ]:
# Optional: download from Colab to your machine
from google.colab import files

files.download(OUTPUT_CSV)